# 03 · Agents and the LangGraph workflow (real Groq models)

```
START ─► supervisor ──Send (in parallel)──► specialist ─► synthesize ─► guardrail ─► END
          RoutePlan                         ReAct agent     FinalAnswer    drops invented
          (structured output)               + its tools     (structured)   citations
```

| Role | Groq model | Why |
|---|---|---|
| supervisor | `openai/gpt-oss-120b` | one small routing call |
| specialist | `openai/gpt-oss-20b` | tool calling, most tokens |
| synthesizer | `openai/gpt-oss-120b` | one structured answer |

The free tier allows 8,000 tokens per minute **per model**, so splitting the roles across two models doubles the budget.

**What you will learn:** each step on its own, then the whole graph, then the guardrail.

In [1]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [2]:
from pharma_agent.config import load_config
from pharma_agent.service import PharmaAgent

cfg = load_config(overrides={"store.path": ":memory:"})
agent = await PharmaAgent.create(cfg)      # real Groq models, tools in-process
for name, s in agent.specialists.items():
    print(f"{name:17s} tools={s.spec.tools}")

label_agent       tools=['search_drug_labels', 'get_adverse_event_counts']
trials_agent      tools=['search_clinical_trials']
literature_agent  tools=['search_pubmed']


## The compiled graph (Mermaid text; paste into https://mermaid.live to draw it)

In [3]:
print(agent.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	specialist(specialist)
	synthesize(synthesize)
	guardrail(guardrail)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	specialist --> synthesize;
	supervisor -.-> specialist;
	synthesize --> guardrail;
	guardrail --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 1 · The supervisor alone: structured output (`RoutePlan`)

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage
from pharma_agent.agents.prompts import SUPERVISOR
from pharma_agent.llm import build_llms
from pharma_agent.schemas import RoutePlan

llms = build_llms(cfg["llm"])
catalog = "\n".join(f"- {n}: {s.spec.description}" for n, s in agent.specialists.items())
question = "What are the main safety warnings for empagliflozin, and are there trials of it in chronic kidney disease?"

plan = await llms.supervisor.with_structured_output(RoutePlan, method="json_schema").ainvoke(
    [SystemMessage(SUPERVISOR.format(catalog=catalog)), HumanMessage(question)])
show(plan)

{
  "reasoning": "The question asks for FDA safety warnings (label_agent) and for information on trials in CKD (trials_agent).",
  "tasks": [
    {
      "agent": "label_agent",
      "instruction": "List the main safety warnings and boxed warnings for empagliflozin from its FDA label."
    },
    {
      "agent": "trials_agent",
      "instruction": "Find registered clinical trials of empagliflozin that involve patients with chronic kidney disease, including phase, status, and sponsor."
    }
  ]
}


## Step 2 · One specialist alone: a ReAct agent calling its tools

In [5]:
text, evidence, calls = await agent.specialists["label_agent"].run("Find the boxed warning and key warnings.", question)
print("Tool calls:", calls)
print("Evidence:", [e.id for e in evidence])
print("\nSummary:\n", text)

Tool calls: [{'tool': 'search_drug_labels', 'args': {'drug': 'empagliflozin', 'limit': 1, 'section': 'warnings'}}]
Evidence: ['fda-label:0fdd0255:warnings']

Summary:
 **Safety warnings for empagliflozin (FDA label)**  
- **Boxed warning**: *None.*  
- **Key warnings**:  
  - **Hypotension and volume depletion** – may occur, especially in patients with renal impairment or on diuretics.  
  - **Genital mycotic infections** – increased risk of vulvovaginal and urinary tract fungal infections.  
  - **Diabetic ketoacidosis** – can present with normal or only mildly elevated glucose levels; monitor for nausea, vomiting, abdominal pain, or malaise.  
  - **Renal impairment** – use with caution; dose adjustment or discontinuation may be required if eGFR falls below 45 mL/min/1.73 m².  
  - **Cardiovascular events** – monitor for signs of heart failure; empagliflozin has shown benefits in heart‑failure trials but may worsen fluid‑balance status.  
  - **Hypoglycemia** – when used with insulin

## Step 3 · The whole graph, and every intermediate result

In [6]:
state = await agent.run_graph(question)
print("Plan:", [(t.agent, t.instruction) for t in state["plan"].tasks])
print("\nTool calls:")
for c in state["tool_calls"]:
    print("  ", c)
print("\nFindings:")
for f in state["findings"]:
    print(f"  [{f.agent}] {f.text[:300]}\n")

Plan: [('label_agent', 'List the main safety warnings and boxed warnings for empagliflozin from its FDA label.'), ('trials_agent', 'Find registered clinical trials of empagliflozin that involve patients with chronic kidney disease, including phase, status, and sponsor.')]

Tool calls:
   {'agent': 'label_agent', 'tool': 'search_drug_labels', 'args': {'drug': 'empagliflozin', 'limit': 1, 'section': 'warnings'}}
   {'agent': 'label_agent', 'tool': 'search_drug_labels', 'args': {'drug': 'empagliflozin', 'limit': 1, 'section': 'warnings'}}
   {'agent': 'label_agent', 'tool': 'search_drug_labels', 'args': {'drug': 'empagliflozin', 'limit': 1, 'section': 'warnings'}}
   {'agent': 'label_agent', 'tool': 'search_drug_labels', 'args': {'drug': 'empagliflozin', 'limit': 1, 'section': 'warnings'}}
   {'agent': 'trials_agent', 'tool': 'search_clinical_trials', 'args': {'condition': 'chronic kidney disease', 'intervention': 'empagliflozin', 'limit': 10, 'status': 'ANY'}}

Findings:
  [label_agent] 

In [7]:
show(state["answer"], 3000)
print("\nGuardrail:", state["guardrail"])

{
  "answer": "The FDA label for empagliflozin (as part of Synjardy) includes a boxed warning for lactic acidosis, especially when combined with metformin. Several ongoing clinical trials are evaluating empagliflozin in various chronic kidney disease (CKD) populations, ranging from pediatric CKD to end‑stage renal disease.",
  "key_points": [
    {
      "text": "The primary safety warning for empagliflozin in the Synjardy label is a boxed warning for lactic acidosis, with post‑marketing cases reporting severe outcomes.",
      "citations": [
        "fda-label:0fdd0255:warnings"
      ]
    },
    {
      "text": "Empagliflozin is being studied in pediatric CKD to assess feasibility and effects on kidney and heart risk factors (Phase 1, recruiting).",
      "citations": [
        "nct:NCT06430684"
      ]
    },
    {
      "text": "Phase 4 trials are recruiting to evaluate empagliflozin’s impact on podocyte injury markers in African‑American veterans with non‑diabetic CKD and diabeti

## Step 4 · The guardrail catches invented citations
LLMs sometimes cite ids that no tool returned. Here we take the real answer and add one made-up id to see
what the guardrail does: it removes the id, counts the point as unsupported and lowers confidence.

In [8]:
from pharma_agent.agents import check_citations, unique_evidence
from pharma_agent.schemas import KeyPoint

tampered = state["answer"].model_copy(deep=True)
tampered.key_points.append(KeyPoint(text="A claim with a made-up source", citations=["pmid:00000000"]))
tampered.confidence = "high"
cleaned, report = check_citations(tampered, unique_evidence(state["evidence"]))
print(report)
print("Confidence after the check:", cleaned.confidence)

evidence_count=11 valid_citations=7 invalid_citations=['pmid:00000000'] repaired_citations=[] unsupported_points=1
Confidence after the check: medium


## Try it yourself
* Ask a literature question, e.g. *"What does research say about metformin in polycystic ovary syndrome?"*
* Change a specialist's `prompt` in `cfg["agents"]`, rebuild with `PharmaAgent.create(cfg)`, and compare.
* Next: **04_api_and_evaluation.ipynb**.